In [4]:
import pandas as pd
import numpy as np
import os
import joblib
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error as mape
import warnings

# ==========================================
# 1. Configuration and Paths
# ==========================================
BASE_PATH = '..'
INPUT_FILE = os.path.join(BASE_PATH, 'includes', 'dados', 'Tabela_consumo_Itapua_120m.csv')
MODEL_SAVE_PATH = 'best_sarima_model.pkl'
LOG_FILE = 'sarima_training_log.csv'

# Suppress warnings for cleaner output during grid search
warnings.filterwarnings("ignore")

def main():
    print("--- Starting SARIMA Training with Lag Window Optimization (1-11) ---")

    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file not found at {INPUT_FILE}")
        return

    # ==========================================
    # 2. Data Loading and Preparation
    # ==========================================
    df = pd.read_csv(INPUT_FILE, sep=';')
    df['AM_REFERENCIA'] = pd.to_datetime(df['AM_REFERENCIA'], format='%Y%m')
    df_aggregated = df.groupby('AM_REFERENCIA')['HCLQTCON'].sum().reset_index()
    df_aggregated = df_aggregated.sort_values(by='AM_REFERENCIA')
    ts = df_aggregated.set_index('AM_REFERENCIA')['HCLQTCON']

    # Splitting Data (70% Train, 30% Test)
    split_idx = int(len(ts) * 0.7)
    train, test = ts[:split_idx], ts[split_idx:]

    # ==========================================
    # 3. Grid Search for Lag Windows (0 to 11)
    # ==========================================
    best_mape = float('inf')
    best_p = 1
    best_model_results = None
    
    results_log = []

    print("\nStarting search for the best AR (p) parameter...")
    for p in range(1, 12):
        try:
            # Testing lag window from 1 to 11
            # Maintaining seasonal order (1, 1, 1, 12) as baseline
            model = SARIMAX(train, 
                            order=(p, 0, 1), 
                            seasonal_order=(1, 1, 1, 12))
            
            model_fit = model.fit(disp=False)
            
            # Forecast on validation set
            predictions = model_fit.forecast(steps=len(test))
            current_mape = mape(test, predictions)
            
            print(f"Lag (p): {p} | MAPE: {current_mape:.4f}")
            results_log.append({'lag_p': p, 'mape': current_mape})

            if current_mape < best_mape:
                best_mape = current_mape
                best_p = p
                best_model_results = model_fit
                
        except Exception as e:
            print(f"Failed to fit model for p={p}: {str(e)}")
            continue

    # ==========================================
    # 4. Saving Best Model and Results
    # ==========================================
    if best_model_results:
        print(f"\nOptimization Complete!")
        print(f"Best Lag (p): {best_p} | Best MAPE: {best_mape:.4f}")
        
        # Saving the fitted model results
        best_model_results.save(MODEL_SAVE_PATH)
        
        # Save logs for reference
        pd.DataFrame(results_log).to_csv(LOG_FILE, index=False)
        print(f"Model saved at: {MODEL_SAVE_PATH}")
        print(f"Training log saved at: {LOG_FILE}")

if __name__ == "__main__":
    main()

--- Starting SARIMA Training with Lag Window Optimization (1-11) ---

Starting search for the best AR (p) parameter...
Lag (p): 1 | MAPE: 0.0575
Lag (p): 2 | MAPE: 0.0632
Lag (p): 3 | MAPE: 0.0553
Lag (p): 4 | MAPE: 0.0552
Lag (p): 5 | MAPE: 0.0531
Lag (p): 6 | MAPE: 0.0540
Lag (p): 7 | MAPE: 0.0542
Lag (p): 8 | MAPE: 0.0550
Lag (p): 9 | MAPE: 0.0530
Lag (p): 10 | MAPE: 0.0548
Lag (p): 11 | MAPE: 0.0548

Optimization Complete!
Best Lag (p): 9 | Best MAPE: 0.0530
Model saved at: best_sarima_model.pkl
Training log saved at: sarima_training_log.csv
